# Análisis exploratorio: Online Retail II

Exploración del dataset de ventas retail online (transacciones, productos, clientes y países).

**Contenido:**
- Carga y primera inspección
- Calidad de datos (nulos, duplicados, tipos)
- Estadísticas descriptivas
- Visualizaciones (distribuciones, series temporales, categorías)
- Conclusiones del EDA

**Dudas**  
¿cómo calcula los revenues?
tendría sentido quitar cuando tenemos identificada una compra?
varios modelos segun comportamiento del cliente?
entender bien como hace la construccion del data set agregado
 


In [ ]:
# Instalamos openpyxl si no lo tenemos (necesario para leer archivos .xlsx)
# Solo hay que ejecutar esta celda una vez
%pip install openpyxl -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 20)
plt.style.use('ggplot')
sns.set_palette('husl')

## 1. Carga de datos

In [ ]:
# -----------------------------------------------------------
# CARGA DE DATOS
# -----------------------------------------------------------
# El archivo Excel tiene 2 hojas (Year 2009-2010 y Year 2010-2011).
# Usamos pd.ExcelFile para leer ambas hojas y las concatenamos
# en un solo DataFrame con pd.concat.
# ignore_index=True reinicia el índice para que sea continuo.
# -----------------------------------------------------------

# Ruta al dataset (relativa al notebook)
DATA_PATH = 'online_retail_II.xlsx'

# Leemos todas las hojas del Excel y las unimos en un solo DataFrame
xl = pd.ExcelFile(DATA_PATH)
print(f'Hojas encontradas: {xl.sheet_names}')

df = pd.concat(
    [pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names],
    ignore_index=True
)

print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')

## 2. Primera inspección

In [ ]:

df.head(10)

In [ ]:

df.info()

# -----------------------------------------------------------
# 3.1 VALORES NULOS
# -----------------------------------------------------------
# isnull().sum() cuenta cuántos nulos hay en cada columna.
# Dividimos entre len(df) para ver el porcentaje.
# Esto nos ayuda a decidir si eliminar filas o imputar valores.
# -----------------------------------------------------------

nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)

pd.DataFrame({
    'Nulos': nulos,
    '% del total': porcentaje_nulos
}).query('Nulos > 0')

In [ ]:
# -----------------------------------------------------------
# 3.2 DUPLICADOS
# -----------------------------------------------------------
# Una fila duplicada = misma factura, producto, cantidad, fecha, precio y cliente.
# Esto puede pasar por errores en la carga de datos.
# Contamos cuántos hay y los eliminaremos en la limpieza.
# -----------------------------------------------------------

n_duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {n_duplicados:,} ({n_duplicados/len(df)*100:.2f}%)')

In [ ]:
# -----------------------------------------------------------
# 3.3 CANCELACIONES
# -----------------------------------------------------------
# Las facturas que empiezan por "C" son cancelaciones.
# Tienen Quantity negativa (devuelven unidades).
# Las identificamos para entender cuántas hay,
# y luego las quitaremos porque no son compras reales.
# -----------------------------------------------------------

# Creamos una columna auxiliar que indica si la factura es una cancelación
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')

n_cancelaciones = df['is_cancelled'].sum()
print(f'Transacciones canceladas: {n_cancelaciones:,} ({n_cancelaciones/len(df)*100:.2f}%)')
print(f'Transacciones normales:   {(~df["is_cancelled"]).sum():,}')

In [ ]:
# -----------------------------------------------------------
# 3.4 LIMPIEZA DEL DATASET
# -----------------------------------------------------------
# Aplicamos 4 filtros de limpieza:
#   1. Eliminar duplicados
#   2. Eliminar cancelaciones (Invoice empieza por "C")
#   3. Eliminar filas sin Customer ID (no podemos modelar sin cliente)
#   4. Eliminar filas con Price <= 0 (no son ventas reales)
#
# Guardamos el resultado en df_clean y mostramos cuántas filas
# quedan después de cada paso.
# -----------------------------------------------------------

print(f'Filas originales: {len(df):,}')

# 1. Quitar duplicados
df_clean = df.drop_duplicates()
print(f'Tras quitar duplicados: {len(df_clean):,}')

# 2. Quitar cancelaciones
df_clean = df_clean[~df_clean['is_cancelled']].copy()
print(f'Tras quitar cancelaciones: {len(df_clean):,}')

# 3. Quitar filas sin Customer ID
df_clean = df_clean.dropna(subset=['Customer ID'])
print(f'Tras quitar sin Customer ID: {len(df_clean):,}')

# 4. Quitar filas con precio <= 0
df_clean = df_clean[df_clean['Price'] > 0].copy()
print(f'Tras quitar Price <= 0: {len(df_clean):,}')

# Eliminamos la columna auxiliar que ya no necesitamos
df_clean = df_clean.drop(columns=['is_cancelled'])

# Convertimos Customer ID a entero (era float por los nulos)
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

print(f'\n>>> Dataset limpio: {len(df_clean):,} filas ({len(df_clean)/len(df)*100:.1f}% del original)')

In [ ]:
## chequear nulos y duplicados

## 4. Estadísticas descriptivas
df.describe()

## 5. Visualizaciones

In [ ]:
### importante gestionarla visualizacion cuando hay outliers 

## 6. Agrupación por cliente (Customer ID)

Por cada cliente: **suma de tickets en el año** (facturación total), **número de compras** (facturas distintas) y **ticket medio** por compra.

## 7. Modelo: probabilidad de recompra

Modelo supervisado para estimar la probabilidad de que un cliente **vuelva a comprar** después de una fecha de corte.

Estrategia:
- Definimos una **fecha de corte** (`cutoff_date`).
- Con los datos **antes** de esa fecha construimos variables por cliente (RFM simplificado: recencia, frecuencia, gasto total, ticket medio, país, etc.).
- Con los datos **después** de la fecha etiquetamos si el cliente **recompró (1)** o **no recompró (0)**.
- Entrenamos una **regresión logística** para predecir esa probabilidad de recompra.

### (Opcional) Búsqueda de hiperparámetros
randomsearch, gridsearch, optuna, 


## 8. Modelo: explicabilidad
indicar las variables más relevantes del modelo. COn Dalex o más sencillo, los propios  tiene manera de obtener las variables:   
Ej. 

from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(model)
plt.show()

